In [ ]:
# Download model from here: https://huggingface.co/collections/meta-llama/llama-31-669fc079a0c406a149a5738f
# change MODEL_PATH
MODEL_PATH = '../models/hf/Meta-Llama-3.1-8B-Instruct'

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    GenerationConfig,
    BitsAndBytesConfig,
    set_seed
)

In [ ]:
# This 4bit quantization will allow loading Llama3.1 8B
# model using ~6GB GPU memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH,
        torch_dtype='auto',
        device_map='auto',
        quantization_config=quantization_config,
        use_cache=True)


In [ ]:
model.generation_config.pad_token_id = model.generation_config.eos_token_id[0]
tokenizer.pad_token = tokenizer.eos_token
#model.generation_config

In [ ]:
PROMPT = "What's the meaning of life?"
input_ids = tokenizer.encode(PROMPT, return_tensors='pt')

In [ ]:
output_ids = model.generate(input_ids.to(model.device), max_new_tokens=100)

In [ ]:
text = tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(text)

In [ ]:
# In local experiments, I usually define simple
# wrapper function.  

def QA (prompt, max_tokens=200):
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    output_ids = model.generate(input_ids.to(model.device), max_new_tokens=max_tokens)
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    print(text)

In [ ]:
QA("Q: What's the meaning of life?\nA:")